In [4]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf


In [5]:
import sys
utils_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
sys.path.append(utils_path)

fixed_path = f'{utils_path}CellCNN'
#cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'
if fixed_path not in sys.path:
    sys.path.append(fixed_path)

In [63]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)


# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation, splitting

from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length

from Dataset_utils import elaborate_predictions, show_hyper

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


# Import datasets from .csv

In [7]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{utils_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    dataset = pd.read_csv(file_path, sep = ';', decimal = ',').astype('float32')
    ALL_DATASETS.append(dataset) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for dataset in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [dataset]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [8]:
for dataset in ALL_DATASETS:
    blast_n = (dataset['IsBlast'] == 1).sum()
    print(f'{blast_n} cells over {len(dataset)} cells')

0 cells over 6558750 cells
0 cells over 2764877 cells
0 cells over 1291729 cells
1348 cells over 843500 cells
170 cells over 1404000 cells
0 cells over 3265250 cells
639 cells over 430869 cells
15 cells over 722000 cells
757 cells over 864000 cells
0 cells over 1947518 cells
308059 cells over 778750 cells
830101 cells over 5510750 cells
72 cells over 208000 cells
0 cells over 2912500 cells
3096 cells over 160500 cells
1449 cells over 3591480 cells
0 cells over 3107000 cells
9147 cells over 637409 cells
227 cells over 2928000 cells
518 cells over 164553 cells
398 cells over 191975 cells
0 cells over 169000 cells
2390 cells over 479000 cells
77 cells over 455000 cells
44697 cells over 160828 cells
1413 cells over 505000 cells


In [9]:
# Show patients donations
print(multiple_donations)

{'11': [3, 4, 5], '12': [6, 7], '1': [8, 9], '2': [10, 11], '3': [12, 13], '4': [14, 15, 16], '6': [17, 18], '7': [19, 20, 21], '8': [22, 23], '9': [24, 25], '13': [0], '14': [1], '15': [2]}


In [10]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


# Elaborate datasets

In [64]:
train_datasets, train_y, val_datasets, val_y, test_datasets_no_labels, test_y, cv_train_datasets, cv_train_y = dataset_elaboration(multiple_donations, ALL_DATASETS, healthy_donors, blast_donors, mixed_donors, n_sub = 2, seed = 30)

Precess starts. Dividing donors...
[4, 2, 3, 0, 1] [1, 0, 2] [3, 1, 0, 4, 2]
Seting Train, Validation and Test idx...
['9', '6', '14', '4', '1'] ['8', '13', '11'] ['12', '2', '15', '7', '3']
Extracting data...
[24, 25, 17, 18, 1, 14, 15, 16, 8, 9]
[22, 23, 0, 3, 4, 5]
[6, 7, 10, 11, 2, 19, 20, 21, 12, 13]
Generating Subsamples...


NameError: name 'blast_donors_idx' is not defined

In [54]:
print(test_y)

[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0]


# CellCNN Restructured

In [33]:
%%time
seed_list = [7359, 9654, 4557, 106, 2615, 6924, 5574, 4552, 2547, 3527]
predictions_list, results_list = run_CellCNN_res(CellCnn, train_datasets, train_y,
                    val_datasets, val_y, 
                    test_datasets_no_labels, 
                    trials = 3,
                    max_epochs = 1, nrun = 3, n_cell = 10, seed_list = seed_list)

Trial 1 started
Seed used: 0
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (20,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (20, 10, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (20, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (12, 2)

# ============================================= #
Run: 0

Seed: 0
X_tr shape: (20, 10, 11)
y_tr shape: (20, 2)
X_v shape: (12, 10, 11)
y_v shape: (12, 2)
Unique values in y_tr: [0. 1.]
1/1 [==============================] - 0s 45ms/step - loss: 0.6821

# ============================================= #
Run: 1

Seed: 1000
X_tr shape: (20, 10, 11)
y_tr shape: (20, 2)
X_v shape: (12, 10, 11)
y_v shape: (12, 2)
Unique values in y_tr: [0. 1.]
1/1 [==============================] - 0s 31ms/step - loss: 0.6690

# ============================================= #
Run: 2

Seed: 2000
X_tr shape: (20, 10, 11)
y_tr shape: (20, 2)
X_v shape: (12, 10, 11)
y_v shape: (12, 2)


In [34]:

tot_trials_res = []
par_list = ['config', 'model_sorted_idx']

for res in results_list:
    needed_results = {}
    for key, value in res.items():
        if key in par_list:
            print(key, value)
            needed_results[key] = value
    tot_trials_res.append(needed_results)
#print(tot_trials_res)

config {'nfilter': [3, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [9, 7, 9], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [5, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 2 1]


In [ ]:
'''

config {'nfilter': [3, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [9, 7, 9], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 1 2]
config {'nfilter': [5, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 2 1]
'''

In [20]:

import json
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\CellCNN\\results\\'
#'''
with open(f'{save_path}new_results_list.json', "w", encoding="utf-8") as f:
    json.dump(tot_trials_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}new_predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(predictions_list, f, default=default_serializer, ensure_ascii=False, indent=2)
#'''
with open(f'{save_path}new_predictions_list.json', "r", encoding="utf-8") as f:
    imported_new_predictions = json.load(f)

with open(f'{save_path}new_results_list.json', "r", encoding="utf-8") as f:
    imported_new_results = json.load(f)

In [21]:
'''
print(results_list[0]['config'])

print(imported_new_results[0])

print(results_list[1]['config'])

print(imported_new_results[1])
'''

{'nfilter': [9, 3, 3], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
{'config': {'nfilter': [9, 3, 3], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}, 'model_sorted_idx': [1, 2, 0]}
{'nfilter': [7, 7, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
{'config': {'nfilter': [7, 7, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}, 'model_sorted_idx': [2, 1, 0]}


In [22]:
best_per_trial = show_hyper(imported_new_results, best_3 = False)
#print(best_per_trial)
new_best_per_trial = show_hyper(results_list, best_3 = False)
#print(new_best_per_trial)

   nfilter  learning_rate  maxpool_percentage
0      3.0            0.0                1.00
1      7.0            0.0                5.00
2      7.0            0.0                5.00
3      3.0            0.0                0.01
4      3.0            0.0                1.00
   nfilter  learning_rate  maxpool_percentage
0      3.0            0.0                1.00
1      7.0            0.0                5.00
2      7.0            0.0                5.00
3      3.0            0.0                0.01
4      3.0            0.0                1.00


In [28]:
pred_phenotype_df, accuracy_list = elaborate_predictions(imported_new_predictions, test_y, results = True)

Trial 0 Accuracy: 0.2727272727272727
Trial 1 Accuracy: 0.2727272727272727
Trial 2 Accuracy: 0.2727272727272727
Trial 3 Accuracy: 0.2727272727272727
Trial 4 Accuracy: 0.2727272727272727
             0   1   2   3   4   5   6   7   8   9   ...  23  24  25  26  27  \
0             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
1             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
2             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
3             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
4             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
True Labels   1   1   1   1   1   1   1   1   1   1  ...   0   1   1   1   1   

             28  29  30  31  32  
0             0   0   0   0   0  
1             0   0   0   0   0  
2             0   0   0   0   0  
3             0   0   0   0   0  
4             0   0   0   0   0  
True Labels   1   1   0   0   0  

[6 rows x 33 co

# Cross Validation Section

In [47]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache

from Dataset_utils import remove_from_cache
remove_from_cache(decache_files)


# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation, splitting

from Dataset_Elaboration_Class import run_CellCNN_res, dataset_elaboration, CellCNN_restructured, CV_CellCNN_restructured, CV_best_acc
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length

from Dataset_utils import elaborate_predictions, show_hyper

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


In [51]:
%%time
from sklearn.model_selection import StratifiedKFold
cv_results, model = CV_CellCNN_restructured(CellCnn, cv_train_datasets, cv_train_y, k = 3, n_cell = 10, n_sub = 3, seed = 42, 
                      max_epochs=1, patience=5, nrun=1)

Fold 1
Creating folds...
Folds Created!
seed: 126000
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (21,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (21, 10, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (21, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (11, 2)

# ============================================= #
Run: 0

Seed: 126000
X_tr shape: (21, 10, 11)
y_tr shape: (21, 2)
X_v shape: (11, 10, 11)
y_v shape: (11, 2)
Unique values in y_tr: [0. 1.]
1/1 [==============================] - 0s 47ms/step - loss: 0.6605

# ============================================= #
Run: 1

Seed: 127000
X_tr shape: (21, 10, 11)
y_tr shape: (21, 2)
X_v shape: (11, 10, 11)
y_v shape: (11, 2)
Unique values in y_tr: [0. 1.]
1/1 [==============================] - 0s 54ms/step - loss: 0.6668

# ============================================= #
Run: 2

Seed: 128000
X_tr shape: (21, 10, 11)
y_tr shape: (21, 2)
X_v shape:

In [52]:

best_fold = CV_best_acc(cv_results) 
#print(best_fold)
# select the cofiguration of the fold that performed best
model.results = cv_results[best_fold]
print(model.results)
# predict using the selected model.results
cv_test_pred = [model.predict(test_datasets_no_labels)]

                    

Best accuracy: 0.6870393753051758
Fold that performed best: 1
{'clustering_result': {'w': array([[ 0.02896731,  0.03440552, -0.0460725 , -0.00896561,  0.0344831 ,
         0.05594864,  0.02122027, -0.02170021, -0.02037784, -0.02512467,
        -0.03262859, -0.00999374, -0.01175863, -0.01015637],
       [ 0.01198534, -0.03724633, -0.04255302,  0.04663949,  0.00918235,
        -0.00817009, -0.00159992,  0.04289434,  0.04211923, -0.01541699,
         0.02599837,  0.00999896,  0.0315453 , -0.05797925],
       [ 0.02146272, -0.01666796,  0.02290733, -0.00325334, -0.00768148,
         0.04310542,  0.01598401, -0.00631354, -0.02357303, -0.01385373,
        -0.01526029, -0.00999817, -0.03214256,  0.00265832],
       [-0.01768421,  0.02699933,  0.0466391 , -0.05979627, -0.02977648,
         0.04016061,  0.03254941,  0.04663018,  0.03874574,  0.00580808,
         0.01220888,  0.00999877,  0.03549931, -0.04267889],
       [-0.02459994,  0.03397779,  0.05266221, -0.01807148,  0.03848724,
        -

In [53]:
cv_tot_folds_res = []
par_list = ['config', 'model_sorted_idx']

for cv_res in cv_results:
    cv_needed_results = {}
    for key, value in cv_res.items():
        if key in par_list:
            print(key, value)
            cv_needed_results[key] = value
    cv_tot_folds_res.append(cv_needed_results)

#print(cv_tot_folds_res)

config {'nfilter': [9, 5, 9], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [1 2 0]
config {'nfilter': [9, 3, 3], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [2 1 0]
config {'nfilter': [5, 9, 7], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
model_sorted_idx [0 2 1]


In [13]:
import json
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\CellCNN\\results\\'
#'''
with open(f'{save_path}cv_new_results_list.json', "w", encoding="utf-8") as f:
    json.dump(cv_tot_folds_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}cv_new_predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(cv_test_pred, f, default=default_serializer, ensure_ascii=False, indent=2)
#'''
with open(f'{save_path}cv_new_predictions_list.json', "r", encoding="utf-8") as f:
    imported_cv_test_pred = json.load(f)

with open(f'{save_path}cv_new_results_list.json', "r", encoding="utf-8") as f:
    imported_cv_new_results = json.load(f)

In [15]:
cv_best_per_fold = show_hyper(imported_cv_new_results, best_3 = False)
print(cv_best_per_fold)

   nfilter  learning_rate  maxpool_percentage
0      7.0            0.0                1.00
1      3.0            0.0                1.00
2      3.0            0.0                5.00
3      7.0            0.0                0.01
4      7.0            0.0                5.00


In [16]:
cv_pred_phenotype_df, cv_accuracy_list = elaborate_predictions( imported_cv_test_pred, test_y, results = True)

Trial 0 Accuracy: 0.3
             0   1   2   3   4   5   6   7   8   9   ...  20  21  22  23  24  \
0             0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   
True Labels   1   1   1   1   1   1   1   1   1   1  ...   0   1   1   1   1   

             25  26  27  28  29  
0             0   0   0   0   0  
True Labels   1   1   0   0   0  

[2 rows x 30 columns]
mean_accuracy over the ten trials: 0.3
accuracy_std over the ten trials: 0.0
